#**Imports, Mount Drive, and Load Data**

In [ ]:
# Imports
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
# Set pandas as the default output for sklearn
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:

fpath = "/content/drive/MyDrive/AXSOSACADEMY/01-Fundamentals/Week04/Data/ames-housing-for-ml.csv"

In [ ]:
df = pd.read_csv(fpath)
df = df.set_index("PID")
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
Index: 2930 entries, 907227090 to 902201120
Data columns (total 35 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   MS Zoning         2930 non-null   object 
 1   Lot Frontage      2440 non-null   float64
 2   Lot Area          2930 non-null   int64  
 3   Street            2930 non-null   object 
 4   Alley             198 non-null    object 
 5   Utilities         2930 non-null   object 
 6   Neighborhood      2930 non-null   object 
 7   Bldg Type         2930 non-null   object 
 8   House Style       2930 non-null   object 
 9   Overall Qual      2930 non-null   int64  
 10  Overall Cond      2930 non-null   int64  
 11  Year Built        2930 non-null   int64  
 12  Year Remodeled    2930 non-null   int64  
 13  Exter Qual        2930 non-null   object 
 14  Exter Cond        2930 non-null   object 
 15  Bsmt Unf Sqft     2929 non-null   float64
 16  Total Bsmnt Sqft  2929 non-null   

,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Utilities,Neighborhood,Bldg Type,House Style,Overall Qual,...,Garage Area,Garage Qual,Garage Cond,Paved Drive,Fence,SalePrice,Month,Year,Total Half Baths,Total Full Baths
PID,,,,,,,,,,,,,,,,,,,,,
907227090,RL,60.0,7200,Pave,NaN,AllPub,CollgCr,1Fam,1Story,5,...,297.0,TA,TA,Y,MnPrv,119900.0,3,2006,0.0,1.0
527108010,RL,134.0,19378,Pave,NaN,AllPub,Gilbert,1Fam,2Story,7,...,576.0,TA,TA,Y,NaN,320000.0,3,2006,1.0,3.0
534275170,RL,NaN,12772,Pave,NaN,AllPub,NAmes,1Fam,1Story,6,...,301.0,TA,TA,Y,NaN,151500.0,4,2007,0.0,1.0
528104050,RL,114.0,14803,Pave,NaN,AllPub,NridgHt,1Fam,1Story,10,...,1220.0,TA,TA,Y,NaN,385000.0,6,2008,0.0,3.0
533206070,FV,32.0,3784,Pave,Pave,AllPub,Somerst,TwnhsE,1Story,8,...,476.0,TA,TA,Y,NaN,193800.0,2,2007,0.0,3.0


#**Preprocess the Data**

**Define X and y, and train test split**

In [ ]:
# Make a list of features to drop
drop_from_model = ['Utilities', # Quasi-constant
                   "Street", # Quasi-constant
                   'MS Zoning', # Stakeholder can't change
                   'Lot Frontage',  # Stakeholder can't change
                   'Lot Area', # Stakeholder can't change
                   'Neighborhood',  # Stakeholder can't change
                   'Year Built'] # Stakeholder can't change
# Define features matrix
X = df.drop(columns = [*drop_from_model,'SalePrice'])
# Define target
y = df['SalePrice']
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)
# Preview training data
X_train.head()

,Alley,Bldg Type,House Style,Overall Qual,Overall Cond,Year Remodeled,Exter Qual,Exter Cond,Bsmt Unf Sqft,Total Bsmnt Sqft,...,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Fence,Month,Year,Total Half Baths,Total Full Baths
PID,,,,,,,,,,,,,,,,,,,,,
905475520,NaN,1Fam,1Story,4,5,1994,TA,TA,0.0,0.0,...,1.0,308.0,TA,TA,N,NaN,8,2007,0.0,1.0
909254010,NaN,1Fam,2Story,7,8,1990,TA,TA,600.0,600.0,...,1.0,215.0,Fa,TA,Y,MnPrv,5,2009,0.0,1.0
531450090,NaN,1Fam,1Story,6,5,1991,TA,TA,78.0,1278.0,...,2.0,496.0,TA,TA,Y,GdWo,6,2008,0.0,3.0
903400040,Pave,1Fam,2Story,6,6,1950,TA,TA,764.0,764.0,...,2.0,520.0,TA,TA,N,GdPrv,7,2007,0.0,1.0
527107130,NaN,1Fam,SLvl,7,5,1997,TA,TA,100.0,384.0,...,2.0,390.0,TA,TA,Y,NaN,6,2009,1.0,2.0


**Make Preprocessing Pipelines for each Feature Type**

In [ ]:
# PREPROCESSING PIPELINE FOR NUMERIC DATA
# Save list of column names
num_cols = X_train.select_dtypes("number").columns
print("Numeric Columns:", num_cols)
# instantiate preprocessors
impute_median = SimpleImputer(strategy='median')
scaler = StandardScaler()
# Make a numeric preprocessing pipeline
num_pipe = make_pipeline(impute_median, scaler)
# Making a numeric tuple for ColumnTransformer
num_tuple = ('numeric', num_pipe, num_cols)

Numeric Columns: Index(['Overall Qual', 'Overall Cond', 'Year Remodeled', 'Bsmt Unf Sqft',
       'Total Bsmnt Sqft', 'Living Area Sqft', 'Bedroom', 'Kitchen',
       'Total Rooms', 'Garage Yr Blt', 'Garage Cars', 'Garage Area', 'Month',
       'Year', 'Total Half Baths', 'Total Full Baths'],
      dtype='object')


In [ ]:
# PREPROCESSING PIPELINE FOR ORDINAL DATA
# Save list of column names
ord_cols = ['Exter Qual','Exter Cond', 'Garage Qual',"Garage Cond"]
print("Ordinal Columns:", ord_cols)
# Create imputer for ordinal data
impute_na_ord = SimpleImputer(strategy='constant', fill_value='NA')
# Making the OrdinalEncoder
# Specifying order of categories for our  Ordinal Qual/Cond Columms
qual_cond_order = ['NA','Po', 'Fa', 'TA', 'Gd', 'Ex']
# Making the list of order lists for OrdinalEncoder
ordinal_category_orders = [qual_cond_order, qual_cond_order,
                           qual_cond_order, qual_cond_order]
ord_encoder = OrdinalEncoder(categories=ordinal_category_orders)
# Making a final scaler to scale category #'s
scaler_ord = StandardScaler()
# Making an ord_pipe
ord_pipe = make_pipeline(impute_na_ord, ord_encoder, scaler_ord)
# Making an ordinal_tuple for ColumnTransformer
ord_tuple = ('ordinal', ord_pipe, ord_cols)

Ordinal Columns: ['Exter Qual', 'Exter Cond', 'Garage Qual', 'Garage Cond']


In [ ]:
# PREPROCESSING PIPELINE FOR ONE-HOT-ENCODED DATA
# Save list of column names
ohe_cols = X_train.select_dtypes('object').drop(columns=ord_cols).columns
print("OneHotEncoder Columns:", ohe_cols)
# Instantiate the individual preprocessors
impute_na = SimpleImputer(strategy='constant', fill_value = "NA")
ohe_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# Make pipeline with imputer and encoder
ohe_pipe = make_pipeline(impute_na, ohe_encoder)
# Making a ohe_tuple for ColumnTransformer
ohe_tuple = ('categorical', ohe_pipe, ohe_cols)

OneHotEncoder Columns: Index(['Alley', 'Bldg Type', 'House Style', 'Central Air', 'Garage Type',
       'Paved Drive', 'Fence'],
      dtype='object')


**Create and Fit the Column Transformer**

In [ ]:
# Create the Column Transformer
col_transformer = ColumnTransformer([num_tuple, ord_tuple, ohe_tuple],
                                    verbose_feature_names_out=False)
# Fit on training data
col_transformer.fit(X_train)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 Index(['Overall Qual', 'Overall Cond', 'Year Remodeled', 'Bsmt Unf Sqft',
       'Total Bsmnt Sqft', 'Living Area Sqft', 'Bedroom', 'Kitchen',
       'Total Rooms', 'Garage Yr Blt', 'Garage Cars', 'Garage Area', 'Month',
       'Year', 'Total Half Baths...
                                 ['Exter Qual', 'Exter Cond', 'Garage Qual',
                                  'Garage Cond']),
                                ('categorical',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='NA',
                                                                strategy='constant')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['Alley', 'Bldg Type', 'House Style', 'Central Air', 'Garage Type',
       'Paved Drive', 'Fence'],
      dtype='object'))],
                  verbose_feature_names_out=False)

**Transform the Data**

In [ ]:
# Transform the training data
X_train_tf = col_transformer.transform(X_train)
# Transform the testing data
X_test_tf = col_transformer.transform(X_test)
# View the processed training data
X_train_tf.head()

,Overall Qual,Overall Cond,Year Remodeled,Bsmt Unf Sqft,Total Bsmnt Sqft,Living Area Sqft,Bedroom,Kitchen,Total Rooms,Garage Yr Blt,...,Garage Type_Detchd,Garage Type_NA,Paved Drive_N,Paved Drive_P,Paved Drive_Y,Fence_GdPrv,Fence_GdWo,Fence_MnPrv,Fence_MnWw,Fence_NA
PID,,,,,,,,,,,,,,,,,,,,,
905475520,-1.453681,-0.502192,0.472590,-1.282165,-2.329907,-1.086450,0.183403,-0.203499,-0.909803,-0.820130,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
909254010,0.648845,2.193205,0.282376,0.093593,-0.996395,-0.525568,0.183403,-0.203499,-0.275453,-2.455923,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
531450090,-0.051997,-0.502192,0.329930,-1.103317,0.510475,-0.395679,0.183403,-0.203499,-0.275453,0.529399,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
903400040,-0.051997,0.396274,-1.619758,0.469633,-0.631901,0.647364,0.183403,-0.203499,0.993247,-1.556237,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
527107130,0.648845,-0.502192,0.615250,-1.052872,-1.476459,0.261635,0.183403,-0.203499,0.358897,0.733873,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


**Instantiate, Fit, Predict, and Evaluate a Default Model**

In [ ]:
def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  rmse = mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- RMSE = {rmse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'RMSE':rmse, 'R^2':r_squared}
      return metrics

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)

  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )

  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set the label as the index
    results_df = results_df.set_index('Label')
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

In [ ]:
# Instantiate a default model with random state for reproducibility
model = DecisionTreeRegressor(random_state = 42)
# Fit the default model on the training data
model.fit(X_train_tf, y_train)
# Use custom function to predict and evaluate
evaluate_regression(model, X_train_tf, y_train, X_test_tf, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 3.778
- MSE = 15,678.198
- RMSE = 15,678.198
- R^2 = 1.000

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 24,500.764
- MSE = 1,609,120,231.370
- RMSE = 1,609,120,231.370
- R^2 = 0.671


#**Define Parameter Dictionary**

In [ ]:
# View parameters that could be tuned
model.get_params()

{'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': 42,
 'splitter': 'best'}

In [ ]:
# Define dictionary of parameters to tune and the values to try
param_grid = {'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, None],
              'min_samples_leaf': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'min_samples_split': [2, 3, 4]}

**Instantiate GridSearchCV**

In [ ]:
#Instantiate GridSearchCV
#👉 “يا GridSearch، خذي هذا الموديلوجربي كل الإعدادات الموجودة في param_gridواستخدمي كل قوة الجهاز وورجيني شو عم يصير”
grid_search = GridSearchCV(model, param_grid, n_jobs = -1, verbose = 1)

**Fit GridSearchCV on Training Data**

In [ ]:
# Fit the Gridsearch on the training data
#جرب كل الاحتمالات وقيّمهم
grid_search.fit(X_train_tf, y_train)

Fitting 5 folds for each of 330 candidates, totalling 1650 fits


GridSearchCV(estimator=DecisionTreeRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, None],
                         'min_samples_leaf': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                         'min_samples_split': [2, 3, 4]},
             verbose=1)

**Obtain the Best Parameters**

In [ ]:
# Obtain the best combination directly
# 👉 هاي أفضل values عطت أعلى performance
grid_search.best_params_

{'max_depth': 10, 'min_samples_leaf': 8, 'min_samples_split': 2}

**Optional - Examine the results in more detail**

In [ ]:
# See results of each combination of parameters
#هون منحول النتائج لجدول (DataFrame)
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_min_samples_leaf,param_min_samples_split,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.010251,0.003136,0.004058,0.000731,1,1,2,"{'max_depth': 1, 'min_samples_leaf': 1, 'min_s...",0.496745,0.506018,0.46953,0.468543,0.490212,0.48621,0.014898,301
1,0.008694,0.000876,0.003971,0.001668,1,1,3,"{'max_depth': 1, 'min_samples_leaf': 1, 'min_s...",0.496745,0.506018,0.46953,0.468543,0.490212,0.48621,0.014898,301
2,0.008967,0.000600,0.003904,0.000928,1,1,4,"{'max_depth': 1, 'min_samples_leaf': 1, 'min_s...",0.496745,0.506018,0.46953,0.468543,0.490212,0.48621,0.014898,301
3,0.009098,0.000839,0.005108,0.002223,1,2,2,"{'max_depth': 1, 'min_samples_leaf': 2, 'min_s...",0.496745,0.506018,0.46953,0.468543,0.490212,0.48621,0.014898,301
4,0.009688,0.002346,0.003422,0.000370,1,2,3,"{'max_depth': 1, 'min_samples_leaf': 2, 'min_s...",0.496745,0.506018,0.46953,0.468543,0.490212,0.48621,0.014898,301


In [ ]:
# Create a list of all columns containing param_
#👉 “اعطيني الأعمدة المهمة بس”
param_cols = cv_results.columns[ cv_results.columns.str.contains('param_')]
# Limit results to the mean_test_score and parameters. Use the * operator to unpack the param_cols list
#👉 “ورجيني أفضل النتائج مرتبة”
results = cv_results.loc[:, ['mean_test_score', *param_cols]].sort_values(by = 'mean_test_score', ascending = False).round(4)
#بعرض أفضل 5 موديلات
results.head()

,mean_test_score,param_max_depth,param_min_samples_leaf,param_min_samples_split
293,0.8140,10,8,4
291,0.8140,10,8,2
292,0.8140,10,8,3
322,0.8136,None,8,3
321,0.8136,None,8,2


#**Define the Best Model**

In [ ]:
# Now define the best version of the model with refit included
#👉 “هذا هو الموديل النهائي”
best_model = grid_search.best_estimator_

#**Make Predictions and Perform Final Evaluation**

In [ ]:
# Use custom function to evalute the model
#👉 “هل الموديل فعلاً كويس ولا لا؟”
evaluate_regression( best_model, X_train_tf, y_train, X_test_tf, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 15,556.335
- MSE = 627,110,888.699
- RMSE = 627,110,888.699
- R^2 = 0.909

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 21,261.349
- MSE = 1,002,114,510.787
- RMSE = 1,002,114,510.787
- R^2 = 0.795
